# Experiment: ACME4 Telemetry Graph Representation Suite

Objective: turn process telemetry into a heterogeneous graph, learn session embeddings, and test whether malicious sessions become locally coherent and reviewable.

Success criteria:
- Beat label-shuffle controls on top-k review recall.
- Show whether graph structure matters with real-edge vs shuffled-edge controls.
- Show whether temporal node encodings improve grouping.
- Keep all labels out of training; labels are used only for evaluation.


## Experiment Design

Task: unsupervised/self-supervised session grouping for analyst triage.

Core methods:
- `graph_stats`: handcrafted session statistics from graph-derived sessions.
- `sage_real_no_time`: HeteroSAGE trained with link prediction on real typed edges.
- `sage_real_time`: HeteroSAGE with temporal process-node encodings.
- `sage_shuffled_time`: same temporal model, but trained on shuffled edges as a topology control.
- `sage_time+stats`: temporal HeteroSAGE fused with graph/session statistics.

Defensibility checks:
- Label-shuffle negative control.
- Graph-stat feature ablation.
- Real-edge vs shuffled-edge control.
- Top-1%, top-5%, top-10% analyst review recall.
- Malicious-neighbor separation.


In [11]:
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from IPython.display import display

warnings.filterwarnings("ignore", message=".*sysctlbyname.*")


In [12]:
SEED = 42
DATA_PATH = "Data/data/ACME4/gold/train-process_uber_summary.parquet"
ROWS = 100_000

RARE_FILE_MIN_DEGREE = 2
RARE_FILE_MAX_DEGREE = 15
SAME_USER_WINDOW = pd.Timedelta(minutes=5)
MAX_SESSION_DURATION = pd.Timedelta(minutes=30)

EPOCHS = 8
HIDDEN = 48
OUT = 48
MAX_POS_PER_EDGE_TYPE = 5_000
K = 15
TOP_FRACTIONS = (0.01, 0.05, 0.10)
EXPERIMENT_SEEDS = (42, 43, 44)

np.random.seed(SEED)
torch.manual_seed(SEED)

## Data Slice

Use the late ACME4 time slice so the experiment is chronological and reproducible. The dataset is sorted by `process_started`, deduplicated by `pid_hash`, and sliced from the tail.


In [13]:
def load_acme_slice(path=DATA_PATH, rows=ROWS):
    df = pd.read_parquet(path)
    df = df.copy()
    df["_time"] = pd.to_datetime(df["process_started"], errors="coerce", utc=True)
    df = (
        df[df["_time"].notna()]
        .sort_values("_time")
        .drop_duplicates("pid_hash", keep="first")
        .tail(rows)
        .reset_index(drop=True)
    )
    return df

late_df = load_acme_slice()

summary = pd.DataFrame([{
    "rows": len(late_df),
    "processes": late_df["pid_hash"].nunique(),
    "red_process_rows": int(late_df["red_team"].fillna(0).astype(int).sum()),
    "start": late_df["_time"].min(),
    "end": late_df["_time"].max(),
}])

display(summary)


,rows,processes,red_process_rows,start,end
0,100000,100000,31,2024-09-18 20:45:15.961034+00:00,2024-09-22 23:57:33.487951+00:00


## Graph Builder

The graph keeps process, file, parent-child, and temporal relations. Known users receive `same_user_time_window` edges; rows without a user receive the distinct `same_host_time_window` fallback. Evaluation sessions use the same identity rule and split after five minutes of inactivity or 30 minutes total duration, preventing transitive multi-day components.

In [14]:
class ACMETelemetryGraphBuilder:
    FEATURE_COLS = [
        "duration_seconds",
        "num_uniq_file_hash",
        "net_total_events",
        "conn_id_count",
        "reg_totals",
    ]

    FEATURE_NAMES = [
        "duration",
        "num_file_hash",
        "net_events",
        "conn_id_count",
        "reg_totals",
    ]

    def __init__(
        self,
        rare_file_min_degree=RARE_FILE_MIN_DEGREE,
        rare_file_max_degree=RARE_FILE_MAX_DEGREE,
        same_user_window=SAME_USER_WINDOW,
        max_session_duration=MAX_SESSION_DURATION,
    ):
        self.rare_file_min_degree = rare_file_min_degree
        self.rare_file_max_degree = rare_file_max_degree
        self.same_user_window = same_user_window
        self.max_session_duration = max_session_duration

    def build(self, df):
        self.df = df.copy().reset_index(drop=True)
        self.df["process_started"] = pd.to_datetime(self.df["process_started"], errors="coerce", utc=True)
        self.process_ids = self.df["pid_hash"].tolist()
        self.node_index = {pid: i for i, pid in enumerate(self.process_ids)}
        self.file_index = {}
        self.parent_child_edges = []
        self.same_user_edges = []
        self.same_host_edges = []
        self.touches_edges = []
        self.file_to_processes = {}

        self._add_parent_child_edges()
        self._add_same_user_edges()
        self._add_same_host_edges()
        self._add_file_edges()

        data = self._to_heterodata()
        self.sessions = self._extract_bounded_sessions()
        data["process"].session_id = self._session_id_tensor(len(self.process_ids), self.sessions)
        return data

    def _process_features(self):
        X = self.df[self.FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        return torch.tensor(X, dtype=torch.float)

    def _process_labels(self):
        y = pd.to_numeric(self.df["red_team"], errors="coerce").fillna(0).astype(int).to_numpy()
        return torch.tensor(y, dtype=torch.long)

    def _add_parent_child_edges(self):
        for row in self.df.itertuples(index=False):
            parent = getattr(row, "parent_pid_hash")
            child = getattr(row, "pid_hash")
            if pd.notna(parent) and parent in self.node_index and child in self.node_index:
                self.parent_child_edges.append((self.node_index[parent], self.node_index[child]))

    def _add_same_user_edges(self):
        valid = self.df["user_name"].notna() & self.df["hostname"].notna()
        for _, group in self.df.loc[valid].groupby(["user_name", "hostname"], dropna=False):
            group = group.sort_values("process_started")
            pids = group["pid_hash"].to_numpy()
            times = group["process_started"].to_numpy()
            for i in range(len(group)):
                src = self.node_index.get(pids[i])
                if src is None:
                    continue
                for j in range(i + 1, len(group)):
                    if times[j] - times[i] > self.same_user_window:
                        break
                    dst = self.node_index.get(pids[j])
                    if dst is not None:
                        self.same_user_edges.append((src, dst))

    def _add_same_host_edges(self):
        valid = self.df["user_name"].isna() & self.df["hostname"].notna()
        for _, group in self.df.loc[valid].groupby("hostname", dropna=False):
            group = group.sort_values("process_started")
            pids = group["pid_hash"].to_numpy()
            times = group["process_started"].to_numpy()
            for i in range(len(group)):
                src = self.node_index.get(pids[i])
                if src is None:
                    continue
                for j in range(i + 1, len(group)):
                    if times[j] - times[i] > self.same_user_window:
                        break
                    dst = self.node_index.get(pids[j])
                    if dst is not None:
                        self.same_host_edges.append((src, dst))


    def _add_file_edges(self):
        file_counts = self.df["filename"].dropna().value_counts()
        rare_files = set(file_counts[
            (file_counts >= self.rare_file_min_degree) &
            (file_counts <= self.rare_file_max_degree)
        ].index)

        for row in self.df.itertuples(index=False):
            fname = getattr(row, "filename")
            pid = getattr(row, "pid_hash")
            if pd.isna(fname) or fname not in rare_files or pid not in self.node_index:
                continue
            if fname not in self.file_index:
                self.file_index[fname] = len(self.file_index)
            pidx = self.node_index[pid]
            fidx = self.file_index[fname]
            self.touches_edges.append((pidx, fidx))
            self.file_to_processes.setdefault(fname, []).append(pidx)

    def _edge_tensor(self, edges):
        return torch.tensor(edges, dtype=torch.long).t().contiguous()

    def _to_heterodata(self):
        data = HeteroData()
        data["process"].x = self._process_features()
        data["process"].y = self._process_labels()
        data["file"].x = torch.ones((len(self.file_index), 1), dtype=torch.float)

        if self.parent_child_edges:
            data["process", "parent_child", "process"].edge_index = self._edge_tensor(self.parent_child_edges)
        if self.same_user_edges:
            data["process", "same_user_time_window", "process"].edge_index = self._edge_tensor(self.same_user_edges)
        if self.same_host_edges:
            data["process", "same_host_time_window", "process"].edge_index = self._edge_tensor(self.same_host_edges)
        if self.touches_edges:
            ei = self._edge_tensor(self.touches_edges)
            data["process", "touches", "file"].edge_index = ei
            data["file", "touched_by", "process"].edge_index = ei.flip(0)
        return data

    def _extract_bounded_sessions(self):
        work = self.df[["hostname", "user_name", "process_started"]].copy()
        work["node_id"] = np.arange(len(work))
        work["session_host"] = work["hostname"]
        missing_host = work["session_host"].isna()
        work.loc[missing_host, "session_host"] = (
            "__missing_host_" + work.loc[missing_host, "node_id"].astype(str)
        )
        work["session_identity"] = work["user_name"].fillna("__host_activity__")
        work = work.sort_values(
            ["session_host", "session_identity", "process_started"],
            kind="mergesort",
        )

        sessions = []
        grouped = work.groupby(
            ["session_host", "session_identity"],
            sort=False,
            dropna=False,
        )
        for _, group in grouped:
            current = []
            session_start = None
            previous_time = None

            for event_time, node_id in group[["process_started", "node_id"]].itertuples(
                index=False,
                name=None,
            ):
                should_split = current and (
                    event_time - previous_time > self.same_user_window
                    or event_time - session_start > self.max_session_duration
                )
                if should_split:
                    sessions.append(current)
                    current = []
                    session_start = None

                if not current:
                    session_start = event_time
                current.append(int(node_id))
                previous_time = event_time

            if current:
                sessions.append(current)

        return sorted(sessions, key=lambda session: session[0])


    def _session_id_tensor(self, n_processes, sessions):
        ids = np.empty(n_processes, dtype=np.int64)
        for sid, session in enumerate(sessions):
            ids[session] = sid
        return torch.tensor(ids, dtype=torch.long)


builder = ACMETelemetryGraphBuilder()
graph_data = builder.build(late_df)
sessions = builder.sessions
process_df = late_df.set_index("pid_hash").reindex(builder.process_ids).reset_index()

session_sizes = np.asarray([len(session) for session in sessions])
session_spans = np.asarray([
    (
        process_df.loc[session, "_time"].max()
        - process_df.loc[session, "_time"].min()
    ).total_seconds() / 60.0
    for session in sessions
])

session_sizes = np.asarray([len(session) for session in sessions])
session_spans = np.asarray([
    (
        process_df.loc[session, "_time"].max()
        - process_df.loc[session, "_time"].min()
    ).total_seconds() / 60.0
    for session in sessions
])

build_summary = pd.DataFrame([{
    "process_nodes": graph_data["process"].num_nodes,
    "file_nodes": graph_data["file"].num_nodes,
    "edge_types": len(graph_data.edge_types),
    "sessions": len(sessions),
    "malicious_sessions": int(sum(any(graph_data["process"].y[n].item() == 1 for n in s) for s in sessions)),
    "median_session_size": float(np.median(session_sizes)),
    "p95_session_size": float(np.quantile(session_sizes, 0.95)),
    "max_session_size": int(session_sizes.max()),
    "p95_session_minutes": float(np.quantile(session_spans, 0.95)),
    "max_session_minutes": float(session_spans.max()),
    "parent_child_edges": len(builder.parent_child_edges),
    "same_user_edges": len(builder.same_user_edges),
    "same_host_edges": len(builder.same_host_edges),
    "touches_edges": len(builder.touches_edges),
}])

display(build_summary)

,process_nodes,file_nodes,edge_types,sessions,malicious_sessions,median_session_size,p95_session_size,max_session_size,p95_session_minutes,max_session_minutes,parent_child_edges,same_user_edges,same_host_edges,touches_edges
0,100000,14,5,3131,19,2.0,137.0,213,29.998144,29.999997,2318,42127,1389585,66


## Evaluation Helpers

The target is malicious session grouping, not supervised classification. Metrics therefore focus on local malicious-neighbor coherence and analyst review recall from unsupervised scores.


In [15]:
def session_labels(data, sessions):
    labels = []
    for session in sessions:
        is_malicious = any(data["process"].y[n].item() == 1 for n in session)
        labels.append("malicious" if is_malicious else "benign")
    return np.asarray(labels)


def knn_scores_and_neighbor_rate(X, labels, k=K):
    X = np.asarray(X, dtype=np.float32)
    y = (np.asarray(labels) == "malicious").astype(int)
    qk = min(k + 1, len(X))
    tree = cKDTree(X)
    dists, idx = tree.query(X, k=qk, workers=-1)
    if qk == 1:
        neighbor_idx = idx[:, None]
        neighbor_dists = dists[:, None]
    else:
        neighbor_idx = idx[:, 1:]
        neighbor_dists = dists[:, 1:]
    neighbor_rate = y[neighbor_idx].mean(axis=1)
    scores = neighbor_dists.mean(axis=1)
    return scores, neighbor_rate


def top_recall_from_scores(scores, labels, fractions=TOP_FRACTIONS):
    y = (np.asarray(labels) == "malicious").astype(int)
    order = np.argsort(scores)[::-1]
    total = int(y.sum())
    rows = []
    for frac in fractions:
        n = max(1, int(np.ceil(len(y) * frac)))
        hits = int(y[order[:n]].sum())
        rows.append({
            "top_fraction": frac,
            "reviewed_sessions": n,
            "malicious_found": hits,
            "total_malicious": total,
            "recall": hits / total if total else np.nan,
            "precision": hits / n,
            "lift": (hits / n) / (total / len(y)) if total else np.nan,
        })
    return pd.DataFrame(rows)


def evaluate_embedding(name, X, labels):
    scores, neighbor_rate = knn_scores_and_neighbor_rate(X, labels)
    y = (np.asarray(labels) == "malicious").astype(int)
    q = top_recall_from_scores(scores, labels)
    row = {
        "method": name,
        "n_sessions": len(y),
        "malicious_sessions": int(y.sum()),
        "mal_neighbor_rate": neighbor_rate[y == 1].mean(),
        "ben_neighbor_rate": neighbor_rate[y == 0].mean(),
        "separation_gap": neighbor_rate[y == 1].mean() - neighbor_rate[y == 0].mean(),
    }
    for _, r in q.iterrows():
        pct = int(r["top_fraction"] * 100)
        row[f"top{pct}_found"] = int(r["malicious_found"])
        row[f"top{pct}_recall"] = r["recall"]
    return row


def label_shuffle_control(X, labels, repeats=50, seed=SEED):
    scores, _ = knn_scores_and_neighbor_rate(X, labels)
    y = np.asarray(labels)
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(repeats):
        shuffled = rng.permutation(y)
        q = top_recall_from_scores(scores, shuffled)
        rows.append({
            "top1_recall": q.loc[q["top_fraction"] == 0.01, "recall"].iloc[0],
            "top5_recall": q.loc[q["top_fraction"] == 0.05, "recall"].iloc[0],
            "top10_recall": q.loc[q["top_fraction"] == 0.10, "recall"].iloc[0],
        })
    out = pd.DataFrame(rows).mean(numeric_only=True).to_frame().T
    out.insert(0, "control", "label_shuffle_mean")
    return out


## Baseline: Graph-Derived Session Statistics

This baseline is intentionally simple: aggregate process-node telemetry within each graph-derived session and append basic structure counts. It is the minimum bar every learned embedding should beat.


In [16]:
GRAPH_STAT_NAMES = [
    "duration_mean", "num_file_hash_mean", "net_events_mean", "conn_id_mean", "reg_totals_mean",
    "duration_std", "num_file_hash_std", "net_events_std", "conn_id_std", "reg_totals_std",
    "session_size", "parent_child_edges", "touches_edges", "same_user_edges", "same_host_edges",
]


def embed_graph_stats(data, sessions):
    n_sessions = len(sessions)
    node_to_session = {}
    for sid, session in enumerate(sessions):
        for node in session:
            node_to_session[node] = sid

    parent_counts = np.zeros(n_sessions, dtype=np.float32)
    touches_counts = np.zeros(n_sessions, dtype=np.float32)
    same_user_counts = np.zeros(n_sessions, dtype=np.float32)
    same_host_counts = np.zeros(n_sessions, dtype=np.float32)

    if ("process", "parent_child", "process") in data.edge_types:
        for u, v in data["process", "parent_child", "process"].edge_index.t().tolist():
            if node_to_session.get(u) == node_to_session.get(v):
                parent_counts[node_to_session[u]] += 1

    if ("process", "touches", "file") in data.edge_types:
        for u, _ in data["process", "touches", "file"].edge_index.t().tolist():
            if u in node_to_session:
                touches_counts[node_to_session[u]] += 1

    if ("process", "same_user_time_window", "process") in data.edge_types:
        for u, v in data["process", "same_user_time_window", "process"].edge_index.t().tolist():
            if node_to_session.get(u) == node_to_session.get(v):
                same_user_counts[node_to_session[u]] += 1

    if ("process", "same_host_time_window", "process") in data.edge_types:
        for u, v in data["process", "same_host_time_window", "process"].edge_index.t().tolist():
            if node_to_session.get(u) == node_to_session.get(v):
                same_host_counts[node_to_session[u]] += 1


    rows = []
    for sid, session in enumerate(sessions):
        feats = data["process"].x[session]
        mean = feats.mean(dim=0).numpy()
        std = feats.std(dim=0, unbiased=False).numpy()
        rows.append(np.concatenate([
            mean,
            std,
            [
                len(session),
                parent_counts[sid],
                touches_counts[sid],
                same_user_counts[sid],
                same_host_counts[sid],
            ],
        ]))

    return np.asarray(rows, dtype=np.float32)


labels_eval = session_labels(graph_data, sessions)
X_graph_stats_raw = embed_graph_stats(graph_data, sessions)
X_graph_stats = StandardScaler().fit_transform(np.nan_to_num(X_graph_stats_raw))
X_node_stats = StandardScaler().fit_transform(np.nan_to_num(X_graph_stats_raw[:, :10]))

baseline_row = evaluate_embedding("graph_stats", X_graph_stats, labels_eval)
display(pd.DataFrame([baseline_row]))

,method,n_sessions,malicious_sessions,mal_neighbor_rate,ben_neighbor_rate,separation_gap,top1_found,top1_recall,top5_found,top5_recall,top10_found,top10_recall
0,graph_stats,3131,19,0.161404,0.005249,0.156155,0,0.0,1,0.052632,5,0.263158


## Graph-Stat Ablation

This tests what part of the graph-stat baseline is actually carrying signal: node telemetry, session size, or edge counts.


In [17]:
ABLATION_GROUPS = {
    "all_graph_stats": np.arange(X_graph_stats_raw.shape[1]),
    "node_means_only": np.arange(0, 5),
    "node_stds_only": np.arange(5, 10),
    "node_stats_only": np.arange(0, 10),
    "session_size_only": np.array([10]),
    "edge_counts_only": np.array([11, 12, 13, 14]),
    "structure_only": np.array([10, 11, 12, 13, 14]),
}

ablation = []
for name, cols in ABLATION_GROUPS.items():
    X = StandardScaler().fit_transform(np.nan_to_num(X_graph_stats_raw[:, cols]))
    ablation.append(evaluate_embedding(name, X, labels_eval))

ablation = pd.DataFrame(ablation).sort_values(
    ["top5_recall", "top1_recall", "separation_gap"],
    ascending=False,
)

display(ablation[[
    "method", "separation_gap",
    "top1_found", "top1_recall",
    "top5_found", "top5_recall",
    "top10_found", "top10_recall",
]])

display(label_shuffle_control(X_graph_stats, labels_eval))

,method,separation_gap,top1_found,top1_recall,top5_found,top5_recall,top10_found,top10_recall
3,node_stats_only,0.156305,0,0.0,5,0.263158,14,0.736842
1,node_means_only,0.138226,0,0.0,4,0.210526,14,0.736842
2,node_stds_only,0.091502,0,0.0,3,0.157895,12,0.631579
0,all_graph_stats,0.156155,0,0.0,1,0.052632,5,0.263158
6,structure_only,0.165036,0,0.0,0,0.000000,0,0.000000
5,edge_counts_only,0.164865,0,0.0,0,0.000000,0,0.000000
4,session_size_only,0.000000,0,0.0,0,0.000000,0,0.000000


,control,top1_recall,top5_recall,top10_recall
0,label_shuffle_mean,0.007368,0.045263,0.091579


## Self-Supervised Heterogeneous Graph Learning

Training task: typed link prediction. The model learns node embeddings that score real graph edges higher than randomly sampled fake edges. Labels are not used during training.

Temporal encoding adds process-node features: normalized event time, hour-of-day sin/cos, day-of-week sin/cos, and log time since the previous process event for the same user/host.


In [18]:
def process_temporal_features(df_proc):
    t = pd.to_datetime(df_proc["process_started"], errors="coerce", utc=True)

    valid = t.notna().to_numpy()
    time_norm = np.zeros(len(df_proc), dtype=float)
    if valid.any():
        secs = t[valid].astype("int64").to_numpy(dtype=float) / 1e9
        span = max(secs.max() - secs.min(), 1.0)
        time_norm[valid] = (secs - secs.min()) / span

    hour = (t.dt.hour.fillna(0) + t.dt.minute.fillna(0) / 60.0).to_numpy(dtype=float)
    dow = t.dt.dayofweek.fillna(0).to_numpy(dtype=float)

    work = df_proc[["hostname", "user_name"]].copy()
    work["_time"] = t
    work["_row"] = np.arange(len(df_proc))
    work = work.sort_values(["hostname", "user_name", "_time"], kind="mergesort")
    delta = work.groupby(["hostname", "user_name"], dropna=False)["_time"].diff().dt.total_seconds()
    work["_delta"] = delta.clip(lower=0, upper=3600).fillna(0)
    delta_aligned = work.sort_values("_row")["_delta"].to_numpy(dtype=float)

    feats = np.column_stack([
        time_norm,
        np.sin(2 * np.pi * hour / 24.0),
        np.cos(2 * np.pi * hour / 24.0),
        np.sin(2 * np.pi * dow / 7.0),
        np.cos(2 * np.pi * dow / 7.0),
        np.log1p(delta_aligned),
    ])
    return StandardScaler().fit_transform(np.nan_to_num(feats))


def make_x_dict(data, df_proc, use_time=False):
    x_dict = {}
    for ntype in data.node_types:
        x = data[ntype].x.float()
        if ntype == "process":
            base = StandardScaler().fit_transform(x.cpu().numpy())
            if use_time:
                base = np.hstack([base, process_temporal_features(df_proc)])
            x = torch.tensor(base, dtype=torch.float)
        x_dict[ntype] = x
    return x_dict


def base_edge_dict(data):
    return {et: data[et].edge_index for et in data.edge_types}


def add_reverse_edges(edge_dict):
    out = dict(edge_dict)
    for src, rel, dst in list(edge_dict.keys()):
        ei = edge_dict[(src, rel, dst)]
        if src == dst:
            out[(dst, f"rev_{rel}", src)] = ei.flip(0)
        else:
            has_reverse = any(et[0] == dst and et[2] == src for et in out)
            if not has_reverse:
                out[(dst, f"rev_{rel}", src)] = ei.flip(0)
    return out


def shuffled_base_edge_dict(data, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    out = {}
    for et in data.edge_types:
        ei = data[et].edge_index
        if ei.shape[1] <= 1:
            out[et] = ei.clone()
            continue
        perm = torch.randperm(ei.shape[1], generator=g)
        out[et] = torch.stack([ei[0].clone(), ei[1, perm].clone()], dim=0)
    return out


class SmallHeteroSAGE(nn.Module):
    def __init__(self, edge_types, hidden=HIDDEN, out=OUT):
        super().__init__()
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((-1, -1), hidden) for et in edge_types}, aggr="sum"),
            HeteroConv({et: SAGEConv((-1, -1), out) for et in edge_types}, aggr="sum"),
        ])

    def forward(self, x_dict, edge_index_dict):
        z = self.convs[0](x_dict, edge_index_dict)
        z = {k: F.relu(v) for k, v in z.items()}
        return self.convs[1](z, edge_index_dict)


def score_edges(z_src, z_dst, edge_index):
    src, dst = edge_index
    return 5.0 * F.cosine_similarity(z_src[src], z_dst[dst], dim=1)


def negative_edges(num_src, num_dst, n_edges, generator):

    return torch.stack([

        torch.randint(0, num_src, (n_edges,), generator=generator),

        torch.randint(0, num_dst, (n_edges,), generator=generator),

    ], dim=0)



def train_sage(name, edge_index_dict, x_dict, seed):

    torch.manual_seed(seed)

    generator = torch.Generator().manual_seed(seed + 10_000)

    model = SmallHeteroSAGE(list(edge_index_dict.keys()))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        opt.zero_grad()
        z = model(x_dict, edge_index_dict)
        losses = []

        for et in graph_data.edge_types:
            src_type, _, dst_type = et
            pos_edge = edge_index_dict[et]
            n_pos = min(MAX_POS_PER_EDGE_TYPE, pos_edge.shape[1])
            if n_pos == 0:
                continue

            perm = torch.randperm(pos_edge.shape[1], generator=generator)[:n_pos]
            pos_edge = pos_edge[:, perm]
            neg_edge = negative_edges(
                graph_data[src_type].num_nodes,
                graph_data[dst_type].num_nodes,
                n_pos,
                generator,
            )

            pos_score = score_edges(z[src_type], z[dst_type], pos_edge)
            neg_score = score_edges(z[src_type], z[dst_type], neg_edge)
            losses.append(
                F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score)) +
                F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
            )

        loss = torch.stack(losses).mean()
        loss.backward()
        opt.step()

        if epoch in {0, EPOCHS - 1}:
            print(f"{name} epoch={epoch:02d} loss={loss.detach().item():.4f}")

    model.eval()
    with torch.no_grad():
        z = model(x_dict, edge_index_dict)
    return z


def mean_pool_sessions(process_embeddings, sessions):
    return np.vstack([process_embeddings[s].mean(axis=0) for s in sessions])


def session_embedding_from_z(z, sessions):
    X = mean_pool_sessions(z["process"].cpu().numpy(), sessions)
    return StandardScaler().fit_transform(np.nan_to_num(X))

## Core Experiment Suite

Each learned method is trained with three fixed random seeds. Reported means and standard deviations therefore capture basic optimization variability instead of relying on one favorable run.

In [19]:
real_edges = add_reverse_edges(base_edge_dict(graph_data))
x_no_time = make_x_dict(graph_data, process_df, use_time=False)
x_time = make_x_dict(graph_data, process_df, use_time=True)

run_rows = []
for seed in EXPERIMENT_SEEDS:
    shuffled_edges = add_reverse_edges(shuffled_base_edge_dict(graph_data, seed=seed))

    z_sage_no_time = train_sage(
        f"sage_real_no_time seed={seed}",
        real_edges,
        x_no_time,
        seed,
    )
    z_sage_time = train_sage(
        f"sage_real_time seed={seed}",
        real_edges,
        x_time,
        seed,
    )
    z_sage_shuffled_time = train_sage(
        f"sage_shuffled_time seed={seed}",
        shuffled_edges,
        x_time,
        seed,
    )

    method_embeddings = {
        "graph_stats": X_graph_stats,
        "node_stats_baseline": X_node_stats,
        "sage_real_no_time": session_embedding_from_z(z_sage_no_time, sessions),
        "sage_real_time": session_embedding_from_z(z_sage_time, sessions),
        "sage_shuffled_time": session_embedding_from_z(z_sage_shuffled_time, sessions),
    }
    method_embeddings["sage_time+stats"] = StandardScaler().fit_transform(
        np.hstack([X_graph_stats, method_embeddings["sage_real_time"]])
    )

    for method, embedding in method_embeddings.items():
        row = evaluate_embedding(method, embedding, labels_eval)
        row["seed"] = seed
        run_rows.append(row)

comparison_runs = pd.DataFrame(run_rows)
metric_cols = [
    "separation_gap",
    "top1_recall",
    "top5_recall",
    "top10_recall",
]
comparison = comparison_runs.groupby("method")[metric_cols].agg(["mean", "std"])
comparison.columns = [f"{metric}_{stat}" for metric, stat in comparison.columns]
comparison = comparison.reset_index().sort_values(
    ["separation_gap_mean", "top10_recall_mean"],
    ascending=False,
)

display(comparison)
display(
    comparison_runs[
        ["method", "seed", "separation_gap", "top1_recall", "top5_recall", "top10_recall"]
    ].sort_values(["method", "seed"])
)

sage_real_no_time seed=42 epoch=00 loss=2.5716
sage_real_no_time seed=42 epoch=07 loss=1.6276
sage_real_time seed=42 epoch=00 loss=1.9934
sage_real_time seed=42 epoch=07 loss=1.0803
sage_shuffled_time seed=42 epoch=00 loss=2.0730
sage_shuffled_time seed=42 epoch=07 loss=1.2511
sage_real_no_time seed=43 epoch=00 loss=2.9363
sage_real_no_time seed=43 epoch=07 loss=1.6637
sage_real_time seed=43 epoch=00 loss=2.1907
sage_real_time seed=43 epoch=07 loss=1.0866
sage_shuffled_time seed=43 epoch=00 loss=2.2433
sage_shuffled_time seed=43 epoch=07 loss=1.2308
sage_real_no_time seed=44 epoch=00 loss=2.7722
sage_real_no_time seed=44 epoch=07 loss=1.6401
sage_real_time seed=44 epoch=00 loss=2.1333
sage_real_time seed=44 epoch=07 loss=1.0811
sage_shuffled_time seed=44 epoch=00 loss=2.2115
sage_shuffled_time seed=44 epoch=07 loss=1.2609


,method,separation_gap_mean,separation_gap_std,top1_recall_mean,top1_recall_std,top5_recall_mean,top5_recall_std,top10_recall_mean,top10_recall_std
2,sage_real_no_time,0.175624,0.010663,0.000000,0.000000,0.000000,0.000000,0.192982,0.030387
1,node_stats_baseline,0.156305,0.000000,0.000000,0.000000,0.263158,0.000000,0.736842,0.000000
0,graph_stats,0.156155,0.000000,0.000000,0.000000,0.052632,0.000000,0.263158,0.000000
5,sage_time+stats,0.119108,0.009232,0.000000,0.000000,0.000000,0.000000,0.105263,0.000000
3,sage_real_time,0.065158,0.012332,0.000000,0.000000,0.017544,0.030387,0.596491,0.109561
4,sage_shuffled_time,0.039583,0.031184,0.105263,0.052632,0.228070,0.060774,0.421053,0.139250


,method,seed,separation_gap,top1_recall,top5_recall,top10_recall
0,graph_stats,42,0.156155,0.000000,0.052632,0.263158
6,graph_stats,43,0.156155,0.000000,0.052632,0.263158
12,graph_stats,44,0.156155,0.000000,0.052632,0.263158
1,node_stats_baseline,42,0.156305,0.000000,0.263158,0.736842
7,node_stats_baseline,43,0.156305,0.000000,0.263158,0.736842
13,node_stats_baseline,44,0.156305,0.000000,0.263158,0.736842
2,sage_real_no_time,42,0.179860,0.000000,0.000000,0.157895
8,sage_real_no_time,43,0.163494,0.000000,0.000000,0.210526
14,sage_real_no_time,44,0.183518,0.000000,0.000000,0.210526
3,sage_real_time,42,0.078088,0.000000,0.000000,0.473684


## Result and Defensible Claim

The bounded construction produced 3,131 sessions from 100,000 processes, with no session longer than 30 minutes or larger than 213 processes. Nineteen sessions contain red-team activity.

**Primary grouping result:** real-edge SAGE without explicit time achieved the strongest malicious-neighbor separation (`0.176 +/- 0.011`), above the node-stat baseline (`0.156`) and shuffled-edge temporal SAGE (`0.040 +/- 0.031`). This supports the claim that the learned graph topology contributes to malicious-session grouping.

**Secondary review result:** node statistics recovered `73.7%` of malicious sessions in the top 10% by distance. Temporal SAGE recovered `59.6% +/- 11.0%`, versus `42.1% +/- 13.9%` for shuffled temporal edges. Time therefore helps review prioritization, but the learned model does not dominate the raw telemetry baseline.

The no-time SAGE model has strong neighbor grouping but weak outlier recall. That is not contradictory: malicious sessions can form coherent neighborhoods without being globally isolated. The defensible conclusion is therefore **relational learning improves grouping**, not that every malicious session becomes an extreme outlier.

Current limitations are the small number of malicious sessions, one ACME time slice, and the same-host fallback required when user identity is absent. External-dataset and temporal-holdout validation remain necessary before claiming generalization.


## Extension Points

Keep these out of the core table until the suite above is stable:
- HGT link prediction as a stronger heterogeneous message-passing model.
- Metapath2Vec / Node2Vec projected-graph baselines.
- Graph autoencoder reconstruction error for benign-normality scoring.
- Agentic edge proposal layer: inspect schema, propose node/edge types, validate edge statistics, build graph config.
- Toponymy-inspired reclustering: label suspicious clusters, summarize behavior, rerank clusters, feed analyst revisions back into clustering.
